***
# Grouping and aggregation with pandas

## Aggregation and reduction

*Split-apply-combine* operations:

1. *Split* data into groups based on some criteria;
2. *Apply* some function to each group separately; and
3. *Combine* the results into a single `DataFrame` or `Series`.

See the pandas
[cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
for an illustration of such operations.

***
### Aggregations of whole Series or DataFrames

Pandas supports the usual set of aggregation functions, e.g.:

-   [`mean()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)
-   [`sum()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)
-   [`std()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html), 
    [`var()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.var.html)
-   [`quantile()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html)
-   [`count()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html)
-   [`min()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.min.html), 
    [`max()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.max.html)

*Example: compute mean of all numerical columns*

-   Use `numeric_only=True` to select numeric columns
-   Missing values are automatically ignored

In [48]:
import pandas as pd

# Relative path to data directory
DATA_PATH = '../../data'

# Path to Titanic passenger data CSV file
file = f'{DATA_PATH}/titanic.csv'

# Read in Titanic passenger data, set PassengerId column as index
df = pd.read_csv(file, index_col='PassengerId')

In [2]:
# Compute the mean for all numerical columns
df.mean(numeric_only=True)

Survived     0.383838
Pclass       2.308642
Age         29.699118
Fare        32.204208
dtype: float64

In [3]:
# Compute average age
df['Age'].mean()

np.float64(29.69911764705882)

In [6]:
import numpy as np
# Compute avg. age with numpy
np.mean(df['Age'])

# Convert to numpy array, then compute mean
np.mean(df['Age'].to_numpy())

np.float64(nan)

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the Titanic dataset to perform the following aggregations:

1. Compute the median age using the column `Age` and the
   [`median()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.median.html)
   method.
2. Compute the fraction of female passengers using the information in column `Sex`.
</div>
<span style="display: none;">YourTurnEnd</span>

In [18]:
median_age = df['Age'].median()
print(f'Median Age: {median_age}')

Median Age: 28.0


In [13]:
df

,Survived,Pclass,Name,Sex,Age,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss Laina",female,26.0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,211536,13.0000,NaN,S
888,1,1,"Graham, Miss Margaret Edith",female,19.0,112053,30.0000,B42,S
889,0,3,"Johnston, Miss Catherine Helen ""Carrie""",female,NaN,W./C. 6607,23.4500,NaN,S


In [21]:
# is_female = df['Sex'] == 'female'.astype(int)
# print(f'Fraction of females: '{is_female.mean()})

(df['Sex'] == 'female').mean()

np.float64(0.35241301907968575)

***
### Aggregations of subsets of data (grouping)

-   We group data using the
    [`groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
    method
    -   Groups are defined based on values of columns or the index, usually a categorical variable or an identifier for a household/individual/firm/period/...
    -   Can group by *multiple* columns or index levels at once
-   We can apply aggregations *by group*

*Example: apply groupby-operations to Titanic data*

-   Tabulate number of observations by class (column `Pclass`) using
    [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
-   Group by passenger class and compute average of numerical columns
    using `numeric_only=True`; missing values are automatically ignored
-   Groups support column indexing

In [22]:
df['Pclass'].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

In [23]:
# Create a groupby object
groups = df.groupby('Pclass')

In [26]:
# Compute averages of numerical columns by passenger class
groups.mean(numeric_only=True)

,Survived,Age,Fare
Pclass,,,
1,0.629630,38.233441,84.154687
2,0.472826,29.877630,20.662183
3,0.242363,25.140620,13.675550


In [29]:
# Manually compute fraction of survivors by class
# Compute unique values of Pclass
pclasses = df['Pclass'].unique()

# Create a series to store pclass-specific fraction of survivors
survived = pd.Series(index=pclasses)

for pclass in pclasses:
    # Restrict to observations for current passenger data
    df_ = df.query(f'Pclass == {pclass}')
    survived.loc[pclass] = df_[survived]

KeyError: "None of [Index([nan, nan, nan], dtype='float64')] are in the [columns]"

#### Built-in aggregations used with `groupby()`

- [`mean()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.mean.html):
    compute average within each group
- [`sum()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.sum.html):
    sum values within each group
- [`std()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.std.html), 
    [`var()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.var.html): 
    within-group standard deviation and variance
-   [`median()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.median.html):
    compute median within each group
- [`quantile()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.quantile.html):
    compute quantiles within each group
- [`size()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.size.html): 
    number of observations in each group
-   [`count()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.count.html):
    number of non-missing observations in each group
-   [`first()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.first.html), 
    [`last()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.last.html): 
    first and last non-missing values in each column within each group
-   [`min()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.min.html), 
    [`max()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.max.html): 
    minimum and maximum elements within a group

See the
[official documentation](https://pandas.pydata.org/docs/user_guide/groupby.html#built-in-aggregation-methods)
for a complete list.

*Example: Number of elements within each group*

-   Use
    [`size()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.size.html)
    to count rows in each group; the result is a `Series`
-   Use
    [`count()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.count.html)
    to count non-missing observations in each column within each group

In [30]:
# Compute number of rows in each group

groups.size()

Pclass
1    216
2    184
3    491
dtype: int64

In [31]:
# Count the number of non-missing observations
groups.count()

,Survived,Name,Sex,Age,Ticket,Fare,Cabin,Embarked
Pclass,,,,,,,,
1,216,216,216,186,216,216,176,214
2,184,184,184,173,184,184,16,184
3,491,491,491,355,491,491,12,491


*Example: Return first non-missing values in each group*

-   Use
    [`first()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.first.html)
    to select the first non-missing value of each column within each group
-   These values can come from different rows

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the Titanic dataset to perform the following aggregations:

1. Compute the average survival rate by sex (stored in the `Sex` column). You need to use the survival indicator stored in the column `Survived` for this.
2. Count the number of passengers aged 50+. Compute the average survival rate by sex for this group.
3. Count the number of passengers below the age of 20 by class and sex. Compute the average survival rate for this group by class and sex.
</div>
<span style="display: none;">YourTurnEnd</span>

In [ ]:
# Part 1: Group by sex
groups_sex = df.groupby('Sex')

groups_sex['Survived'].mean()

df.groupby('Sex')['Survived'].mean()


Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [45]:
# Part 2:
df_age50plus = df.query('Age >= 50')
print(f'Number of passengers ages 50+: {len(df_age50plus)}')
df_age50plus.groupby('Sex')['Survived'].mean()

Number of passengers ages 50+: 74


Sex
female    0.909091
male      0.134615
Name: Survived, dtype: float64

In [46]:
# Part 3:
df_age20max = df.query('Age <= 20')
print(f'Number of observation with ages <= 20: {len(df_age20max)}')
df_age20max.groupby(['Pclass', 'Sex'])['Survived'].mean()

Number of observation with ages <= 20: 179


Pclass  Sex   
1       female    0.928571
        male      0.571429
2       female    1.000000
        male      0.526316
3       female    0.510638
        male      0.197368
Name: Survived, dtype: float64

***
#### Writing custom aggregations

-   Sometimes we want to use aggregations *not* implemented in pandas
-   Use
    [`agg()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html)
    on grouped data to perform custom aggregations
-   `agg()` operates on one column at a time

    -   Use
        [`apply()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.apply.html)
        for statistics that require values from several columns
    -   `apply()` passes each group's `DataFrame` to the custom function,
        excluding grouping columns

*Example: compute mean via `agg()`*

-   Compute mean age by class (`Pclass`)

In [ ]:
# Compute avg age in a complicated way
df.groupby('Pclass')['Age'].agg('mean')

Pclass
1    38.233441
2    29.877630
3    25.140620
Name: Age, dtype: float64

In [50]:
df.groupby('Pclass')['Age'].mean()

Pclass
1    38.233441
2    29.877630
3    25.140620
Name: Age, dtype: float64

*Example: custom aggregation*

-   Compute fraction of passengers aged 40+ by class among those with known ages
-   Use `dropna()` to exclude passengers with unknown ages from the denominator

In [51]:
# Fraction of passengers aged 40+
(df['Age'] >= 40).mean()

np.float64(0.1829405162738496)

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 1 to 891
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    str    
 3   Sex       891 non-null    str    
 4   Age       714 non-null    float64
 5   Ticket    891 non-null    str    
 6   Fare      891 non-null    float64
 7   Cabin     204 non-null    str    
 8   Embarked  889 non-null    str    
dtypes: float64(2), int64(2), str(5)
memory usage: 62.8 KB


In [53]:
# Compute fraction of passenger aged 40+, for known ages
(df['Age'].dropna() >= 40).mean()

np.float64(0.22829131652661064)

In [ ]:
def fcn(series):
    """
    Return fraction aged 40+ for observations nonmissing age
    """
    return (series.dropna() >= 40).mean()

In [55]:
df.groupby('Pclass')['Age'].agg(fcn)

Pclass
1    0.435484
2    0.213873
3    0.126761
Name: Age, dtype: float64

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the Titanic dataset to perform the following aggregation:

- Group passengers by embarkation port (`Embarked`). Use
[`agg()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html)
with a `lambda` expression to find the most common passenger class (`Pclass`) for each
port. In statistics, the most common value is called the *mode* of a distribution.

*Hint:* You can use the
[`mode()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html)
method of a Series (e.g., `x.mode().iloc[0]`) or inspect the index of
[`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html).

*Note:* When we group by `Embarked`, pandas excludes the two passengers whose
embarkation port is missing.

</div>
<span style="display: none;">YourTurnEnd</span>

In [57]:
df.groupby('Embarked')['Pclass'].mode()

AttributeError: 'SeriesGroupBy' object has no attribute 'mode'

***
#### Applying multiple functions at once

-  Applying multiple functions to a **single** column

    -   Functions are passed in as a *list* of strings or callables

-  Applying multiple functions to **multiple** columns

    -   Use so-called *named aggregation*
    -   Each keyword names an output column; its value is a
        `(column, operation)` tuple:

    ```python
    groups.agg(
        new_column_name1=('column_name1', 'operation1'),
        new_column_name2=('column_name2', 'operation2'),
        ...
    )
    ```


*Example: Applying multiple functions to a **single** column*

-   Compute mean *and* median age by class

In [58]:
groups = df.groupby('Pclass')
age_mean = groups['Age'].mean()
age_median = groups['Age'].median()

In [60]:
df.groupby('Pclass')['Age'].agg(['mean', 'median'])

,mean,median
Pclass,,
1,38.233441,37.0
2,29.877630,29.0
3,25.140620,24.0


*Example: Applying multiple functions to **multiple** columns*

-   Compute the following by class in a single call to `agg()`:
    1.  Mean age
    2.  Maximum fare (column `Fare`)

In [64]:
df.groupby('Pclass').agg(
    average_age = ('Age', 'mean'), 
    max_fare = ('Fare', 'max'),
    median_age = ('Age', 'median'))

,average_age,max_fare,median_age
Pclass,,,
1,38.233441,512.3292,37.0
2,29.877630,73.5000,29.0
3,25.140620,69.5500,24.0


<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the Titanic dataset to perform the following aggregations:

1. Compute the minimum, maximum, and average age by embarkation port (stored in
   the column `Embarked`) in a single `agg()` operation.
   There are several ways to solve this problem.
2. Compute the number of passengers, the average age, and the fraction of women
   by embarkation port in a single `agg()` operation.
   *Hint:* To compute the fraction of women, you can either use a `lambda` expression
   or first create a numerical indicator variable for females
   (as we did in the workshop).
</div>
<span style="display: none;">YourTurnEnd</span>

In [78]:
df.groupby('Embarked').agg(minimum_age = ('Age', 'min'), maximum_age = ('Age', 'max'), average_age = ('Age', 'mean'))

,minimum_age,maximum_age,average_age
Embarked,,,
C,0.42,71.0,30.814769
Q,2.00,70.5,28.089286
S,0.67,80.0,29.445397


In [88]:
df.groupby('Embarked').agg(
    num = ('Embarked', 'size'),
    avg_age=('Age', 'mean'),
    women=('Sex', lambda x: (x == 'female').mean())
    )

,num,avg_age,women
Embarked,,,
C,168,30.814769,0.434524
Q,77,28.089286,0.467532
S,644,29.445397,0.315217


***
## Transformations

-   Same principle as aggregations, but number of observations remains unchanged
-   Useful for computations that involve individual and aggregate data
-   Performed with
    [`transform()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.transform.html)

*Example: assign group-specific mean*

- Compute average fare by class, assign result to each observation

*Example: Deviation from average fare*

-   For each passenger, compute the relative difference between their fare and
    the average fare in their class: `fare / class_mean - 1`

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the Titanic dataset to perform the following transformation:

1. Compute the *excess* fare paid by each passenger compared to the minimum fare
   by embarkation port and class, i.e., compute $Fare - min(Fare)$
   by port and class.
</div>
<span style="display: none;">YourTurnEnd</span>

***
## Resampling and aggregation

### Reminder: Working with time series data (lecture 2)

-   Pandas indices can be date or datetime data types
-   Use
    [`date_range()`](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html)
    to create a range of dates
-   Use
    [`to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
    to convert existing data to datetime format

*Example: Creating a date index*

-   Create a demo dataset of daily observations for the first 3 months of 2025

In [2]:
# Start and end dates used for demo data set
start = '2025-01-01'
end = '2025-03-31'

*Example: Select particular date*

-   Select observation from January 1, 2025

*Example: Select date range*

- Select first 5 days in January 2025

*Example: Use a partial index*

- Select all of January 2025

***
### Resampling time series data

-   Use
    [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)
    to aggregate time series data to lower frequency
-   Argument to `resample()` determines frequency and index of resulting data, e.g.:

    -   `'YE'` aggregation to years, index is end of year
    -   `'QE'` aggregation to quarters, index is end of quarter
    -   `'ME'` aggregation to months, index is end of month
    -   `'W'` aggregation to weeks, ending and labeled on Sunday

*Example: Resampling the NASDAQ index*

-   Compute monthly means of daily observations using `resample('ME').mean()`
-   Return the last non-missing observation of each week using
    `resample('W').last()`

In [89]:
# Path to NASDAQ data file
file = f'{DATA_PATH}/stockmarket/NASDAQ.csv'

# Read in NASDAQ data, set Date column as index
df = pd.read_csv(file, index_col='Date', parse_dates=True)

# Keep observations for 2025
df = df.loc['2025']

In [93]:
df.loc['2025-06-30']

NASDAQ    20369.7
Name: 2025-06-30 00:00:00, dtype: float64

In [99]:
# Compute the monthly avg of nasdaq in 2025
df.resample('QE').mean()

,NASDAQ
Date,
2025-03-31,18951.465000
2025-06-30,18309.732258
2025-09-30,21441.934375
2025-12-31,23136.193651


<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Use the daily NASDAQ data for 2025 and compute the percentage change from the first to the last trading day within each month.
</div>
<span style="display: none;">YourTurnEnd</span>

In [101]:
# Resample to monthly frequency
resampler = df.resample('ME')
returns = (resampler.last() - resampler.first()) / resampler.first()* 100.0